# Guia Completo: Knowledge Distillation de Classificadores de Imagens

**MO434 - Aprendizado Profundo para Visao Computacional**

---

## Objetivo do Projeto

Treinar um modelo **student** leve que imita as representacoes internas de um modelo **teacher**
pre-treinado (VGG, ResNet, ConvNeXt), de forma que o student consiga classificar imagens com
muito menos parametros e operacoes (GFLOPs) do que o teacher.

## As 3 Fases

```
Fase 1 -> Treinar o CLASSIFICADOR do teacher (encoder congelado)
Fase 2 -> Treinar ENCODER + PREDITOR do student (imitacao de features do teacher)
Fase 3 -> Testar student com o MESMO classificador treinado na Fase 1
```

## As 5 Perguntas de Pesquisa

| # | Pergunta | O que implementar |
|---|----------|------------------|
| Q1 | Qual teacher transfere melhor? | Comparar VGG-16, ResNet-50, ConvNeXt-S |
| Q2 | Student deve prever pre-GAP ou post-GAP? | Duas variantes de preditor |
| Q3 | Qual a melhor arquitetura do student? | Plain CNN vs Depthwise vs Mini-ResNet |
| Q4 | Qual a melhor funcao de perda? | Ablacao de alpha em L = alpha*MSE + (1-alpha)*CE |
| Q5 | O que aprender da literatura de KD? | Implementar RKD e comparar com baseline |

---

## Dicas de Datasets

Voce precisa de **pelo menos 2 datasets** para comparar. Recomendacoes:

| Dataset | Classes | Imagens | Tamanho | Quando usar |
|---------|---------|---------|---------|-------------|
| **CIFAR-10** | 10 | 60K (32x32) | ~160MB | Debug rapido, valida o codigo |
| **CIFAR-100** | 100 | 60K (32x32) | ~160MB | **Recomendado para Q1-Q4**: diferenca entre teachers fica mais evidente |
| **STL-10** | 10 | 13K (96x96) | ~2.5GB | Regime de poucos dados |
| **Tiny-ImageNet** | 200 | 100K (64x64) | ~236MB | Melhor se tiver GPU boa |
| **Flowers-102** | 102 | 8K variado | ~330MB | Bom para testar ConvNeXt |
| **Food-101** | 101 | 101K variado | ~5GB | Desafiador, rico em textura |

**Estrategia recomendada:**
- Desenvolvimento/debug: CIFAR-10 (loop rapido, ~2 min/epoch na GPU)
- Experimentos principais (Q1-Q4): CIFAR-100 (mais classes = diferencas mais evidentes)
- Validacao cruzada: STL-10 (confirma que resultados generalizam)

> **Por que CIFAR-100 para Q1?** Com 100 classes o teacher precisa de representacoes
> mais ricas, entao a qualidade do backbone importa muito mais. No CIFAR-10 todos os
> teachers tendem a saturar e as diferencas ficam pequenas.


---
## Secao 0 - Instalacao e Importacoes

Execute esta celula uma vez para instalar as dependencias.


In [ ]:
# instalacao das dependencias do projeto
# descomente conforme necessario

# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
# !pip install fvcore torchmetrics matplotlib seaborn pandas tqdm


In [ ]:
# importacoes gerais do projeto
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from copy import deepcopy
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets, models

# verificacao de dispositivo (gpu ou cpu)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"dispositivo em uso: {device}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")
    print(f"memoria gpu: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


---
## Secao 1 - Preparacao dos Datasets

### Por que usar 2 datasets?

O enunciado exige multiplos datasets para garantir que os resultados sejam
**generalizaveis** e nao especificos de um unico dominio.

### Pre-processamento para modelos pre-treinados no ImageNet

Todos os backbones (VGG, ResNet, ConvNeXt) foram treinados com imagens normalizadas
com a **media e desvio padrao do ImageNet**. E obrigatorio usar os mesmos valores:

```
media  = [0.485, 0.456, 0.406]
desvio = [0.229, 0.224, 0.225]
```

Alem disso, os modelos esperam entradas de **224x224 pixels**.


In [ ]:
# parametros de normalizacao do imagenet (obrigatorio para modelos pre-treinados)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# transformacoes padrao (validacao e teste, sem augmentacao)
transform_val = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# transformacoes com augmentacao (somente para o conjunto de treino)
# a augmentacao aumenta a diversidade sem coletar novos dados
transform_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("transformacoes definidas com sucesso!")


In [ ]:
# ── dataset 1: cifar-100 ──────────────────────────────────────────────────────
# 100 classes, 60.000 imagens de 32x32 pixels
# ideal para comparar teachers (Q1) e funcoes de perda (Q4)

cifar100_train_raw = datasets.CIFAR100(root='./data', train=True,  download=True, transform=transform_train)
cifar100_val_raw   = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_val)

# divisao: 80% treino, 20% validacao do conjunto original de treino
n_train = int(0.8 * len(cifar100_train_raw))
n_val   = len(cifar100_train_raw) - n_train
cifar100_train, cifar100_val = random_split(cifar100_train_raw, [n_train, n_val])

# loaders com pin_memory=True para transferencia cpu->gpu mais rapida
loader_c100_train = DataLoader(cifar100_train,    batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
loader_c100_val   = DataLoader(cifar100_val,      batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
loader_c100_test  = DataLoader(cifar100_val_raw,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"cifar-100 -> treino: {len(cifar100_train):,} | val: {len(cifar100_val):,} | teste: {len(cifar100_val_raw):,}")

# ── dataset 2: stl-10 ─────────────────────────────────────────────────────────
# 10 classes, 13.000 imagens de 96x96 pixels
# testa regime de poucos dados com imagens maiores

stl10_train = datasets.STL10(root='./data', split='train', download=True, transform=transform_train)
stl10_test  = datasets.STL10(root='./data', split='test',  download=True, transform=transform_val)

n_stl_train = int(0.8 * len(stl10_train))
n_stl_val   = len(stl10_train) - n_stl_train
stl10_tr, stl10_vl = random_split(stl10_train, [n_stl_train, n_stl_val])

loader_stl_train = DataLoader(stl10_tr,   batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
loader_stl_val   = DataLoader(stl10_vl,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
loader_stl_test  = DataLoader(stl10_test, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"stl-10    -> treino: {len(stl10_tr):,} | val: {len(stl10_vl):,} | teste: {len(stl10_test):,}")

# mapa de configuracoes por dataset (facilita alternar nos experimentos)
DATASETS = {
    'cifar100': {
        'n_classes': 100,
        'train': loader_c100_train,
        'val':   loader_c100_val,
        'test':  loader_c100_test,
    },
    'stl10': {
        'n_classes': 10,
        'train': loader_stl_train,
        'val':   loader_stl_val,
        'test':  loader_stl_test,
    },
}


In [ ]:
# visualizacao: amostras de imagens dos dois datasets

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
mean_t = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std_t  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

# cifar-100: primeira linha
imgs_c, labels_c = next(iter(loader_c100_val))
for i in range(8):
    img = (imgs_c[i] * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"cls {labels_c[i].item()}", fontsize=8)
    axes[0, i].axis('off')
axes[0, 0].set_ylabel("CIFAR-100", fontsize=10, rotation=90)

# stl-10: segunda linha
imgs_s, labels_s = next(iter(loader_stl_val))
for i in range(8):
    img = (imgs_s[i] * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"cls {labels_s[i].item()}", fontsize=8)
    axes[1, i].axis('off')
axes[1, 0].set_ylabel("STL-10", fontsize=10, rotation=90)

plt.suptitle("Amostras dos datasets (normalizacao invertida para visualizacao)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Secao 2 - Modelo Teacher

### Conceito: O papel do Teacher no KD

O **teacher** e um backbone pre-treinado no ImageNet cujo encoder esta **congelado**
(pesos nunca atualizados). Apenas o classificador final e treinado no novo dataset.

### Estrutura de um backbone moderno

```
Imagem [B, 3, 224, 224]
    |
[ENCODER CONGELADO]   <- pesos ImageNet, nunca atualizados
    |
Feature map pre-GAP: [B, C, 7, 7]
    |
Global Average Pooling (GAP)
    |
Vetor post-GAP: [B, C]
    |
[CLASSIFICADOR TREINAVEL]   <- so esta parte e treinada na Fase 1
    |
Logits: [B, n_classes]
```

### Teachers comparados no projeto

| Teacher | Parametros | GFLOPs | dim post-GAP | Caracteristica |
|---------|------------|--------|--------------|----------------|
| VGG-16 | 138M | 15.5 | 512 | Classico, sem skip connections |
| ResNet-50 | 25M | 4.1 | 2048 | Baseline mais estudado em KD |
| ConvNeXt-Small | 50M | 8.7 | 768 | Moderno, melhor accuracy |


In [ ]:
# wrapper do teacher: encapsula encoder + gap + classificador de forma modular
class TeacherWrapper(nn.Module):
    # encapsula qualquer backbone pre-treinado expondo:
    #   encoder: extrai feature maps (sem gap)
    #   gap:     global average pooling
    #   classifier: camada linear final
    # permite extrair pre-gap e post-gap de forma uniforme

    def __init__(self, backbone_name: str, n_classes: int):
        super().__init__()
        self.backbone_name = backbone_name
        self._build(backbone_name, n_classes)

    def _build(self, name, n_classes):
        if name == 'vgg16':
            base = models.vgg16(weights='IMAGENET1K_V1')
            self.encoder    = base.features           # saida: [B, 512, 7, 7]
            self.gap        = nn.AdaptiveAvgPool2d(1) # saida: [B, 512, 1, 1]
            self.feat_dim   = 512
            # substitui o classificador original por um adequado ao novo dataset
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(512, n_classes)
            )

        elif name == 'resnet50':
            base = models.resnet50(weights='IMAGENET1K_V2')
            # remove gap e fc originais, mantemos so as conv layers
            self.encoder    = nn.Sequential(*list(base.children())[:-2])  # saida: [B, 2048, 7, 7]
            self.gap        = nn.AdaptiveAvgPool2d(1)
            self.feat_dim   = 2048
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(2048, n_classes)
            )

        elif name == 'convnext_small':
            base = models.convnext_small(weights='IMAGENET1K_V1')
            self.encoder    = base.features           # saida: [B, 768, 7, 7]
            self.gap        = nn.AdaptiveAvgPool2d(1)
            self.feat_dim   = 768
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.LayerNorm(768),
                nn.Linear(768, n_classes)
            )
        else:
            raise ValueError(f"backbone '{name}' nao suportado. use: vgg16, resnet50, convnext_small")

    def freeze_encoder(self):
        # congela todos os parametros do encoder
        # chamada obrigatoria antes da fase 1
        for p in self.encoder.parameters():
            p.requires_grad = False
        print(f"encoder {self.backbone_name} congelado.")

    def get_pre_gap(self, x):
        # retorna feature map espacial antes do gap: [B, C, 7, 7]
        return self.encoder(x)

    def get_post_gap(self, x):
        # retorna vetor comprimido apos o gap: [B, C]
        feat = self.encoder(x)
        return self.gap(feat).flatten(1)

    def forward(self, x):
        # forward completo: encoder -> gap -> classificador -> logits
        feat   = self.encoder(x)
        pooled = self.gap(feat).flatten(1)
        return self.classifier(pooled)


# teste rapido de shapes para cada teacher
print("verificando shapes dos teachers...")
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224)
    for name in ['vgg16', 'resnet50', 'convnext_small']:
        t      = TeacherWrapper(name, n_classes=100)
        t.eval()
        pre    = t.get_pre_gap(dummy)
        post   = t.get_post_gap(dummy)
        logits = t(dummy)
        params = sum(p.numel() for p in t.parameters()) / 1e6
        print(f"  {name:16s} | pre-gap: {str(pre.shape):22s} | post-gap: {str(post.shape):14s} | params: {params:.1f}M")


---
## Secao 3 - Fase 1: Treinar o Classificador do Teacher

### O que fazemos aqui?

O encoder do teacher ja sabe extrair features (pre-treinado no ImageNet).
Mas o classificador original foi treinado para 1000 classes do ImageNet, nao
para as classes do nosso dataset.

Na Fase 1:
1. Congelamos o encoder (nenhum gradiente passa por ele)
2. Trocamos a ultima camada do classificador por uma nova com `n_classes` saidas
3. Treinamos **apenas o classificador** com Cross-Entropy Loss

### Por que isso e critico para o KD?

O classificador treinado na Fase 1 e **reutilizado exatamente** na Fase 3 para
avaliar o student. Isso forca o student a produzir features que estejam
**no mesmo espaco** do teacher.

### Funcao de perda: Cross-Entropy

$$L_{CE} = -\frac{1}{N}\sum_{i=1}^{N} \log(\hat{y}_{i, c_i})$$

onde $\hat{y}_{i, c_i}$ e a probabilidade predita para a classe correta $c_i$.


In [ ]:
# funcoes de treino e validacao (padrao do curso mo434)

def treinar_batch(model, dados, optimizer, device):
    # executa um passo de treino: forward -> loss -> backward -> atualizar pesos
    # retorna (loss, acuracia) do batch
    model.train()
    imgs, labels = dados
    imgs, labels = imgs.to(device), labels.to(device)

    optimizer.zero_grad()                    # 1. zera gradientes acumulados
    logits = model(imgs)                     # 2. forward pass
    loss   = F.cross_entropy(logits, labels) # 3. cross-entropy loss
    loss.backward()                          # 4. backpropagation
    optimizer.step()                         # 5. atualiza pesos

    acc = (logits.argmax(1) == labels).float().mean().item()
    return loss.item(), acc


@torch.no_grad()
def validar_batch(model, dados, device):
    # avalia o modelo sem computar gradientes (mais rapido e menos memoria)
    # model.eval() desativa dropout e usa estatisticas fixas do batchnorm
    model.eval()
    imgs, labels = dados
    imgs, labels = imgs.to(device), labels.to(device)
    logits = model(imgs)
    loss   = F.cross_entropy(logits, labels)
    acc    = (logits.argmax(1) == labels).float().mean().item()
    return loss.item(), acc


def loop_treino(model, loader_train, loader_val, optimizer, scheduler, n_epochs, device, descricao=""):
    # loop de treino completo com early stopping simples e registro de metricas
    historico  = defaultdict(list)
    melhor_acc = 0.0
    melhor_pesos = None

    for epoch in range(1, n_epochs + 1):
        # --- treino ---
        losses_tr, accs_tr = [], []
        for dados in tqdm(loader_train, desc=f"[{descricao}] epoca {epoch}/{n_epochs}", leave=False):
            l, a = treinar_batch(model, dados, optimizer, device)
            losses_tr.append(l); accs_tr.append(a)

        # --- validacao ---
        losses_vl, accs_vl = [], []
        for dados in loader_val:
            l, a = validar_batch(model, dados, device)
            losses_vl.append(l); accs_vl.append(a)

        loss_tr = np.mean(losses_tr);  acc_tr = np.mean(accs_tr)
        loss_vl = np.mean(losses_vl);  acc_vl = np.mean(accs_vl)

        historico['loss_tr'].append(loss_tr);  historico['acc_tr'].append(acc_tr)
        historico['loss_vl'].append(loss_vl);  historico['acc_vl'].append(acc_vl)

        if scheduler:
            scheduler.step()

        # salva melhores pesos (early stopping simples)
        if acc_vl > melhor_acc:
            melhor_acc   = acc_vl
            melhor_pesos = deepcopy(model.state_dict())

        if epoch % 5 == 0 or epoch == 1:
            print(f"  epoca {epoch:3d} | loss_tr={loss_tr:.4f} acc_tr={acc_tr:.3f} | "
                  f"loss_vl={loss_vl:.4f} acc_vl={acc_vl:.3f}")

    # restaura melhores pesos
    if melhor_pesos:
        model.load_state_dict(melhor_pesos)
    print(f"  melhor acc validacao: {melhor_acc:.4f}")
    return historico


def plotar_curvas(historico, titulo=""):
    # plota curvas de loss e acuracia lado a lado
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(historico['loss_tr'], label='treino')
    ax1.plot(historico['loss_vl'], label='validacao')
    ax1.set_title(f'loss - {titulo}'); ax1.set_xlabel('epoca')
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(historico['acc_tr'], label='treino')
    ax2.plot(historico['acc_vl'], label='validacao')
    ax2.set_title(f'acuracia - {titulo}'); ax2.set_xlabel('epoca')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


In [ ]:
# ── fase 1: treinar classificador de cada teacher ──────────────────────────

DATASET_ESCOLHIDO = 'cifar100'   # troque por 'stl10' para o segundo dataset
N_CLASSES         = DATASETS[DATASET_ESCOLHIDO]['n_classes']
EPOCHS_FASE1      = 15           # 15 epocas sao suficientes: o encoder ja e bom
LR_FASE1          = 1e-3

# dicionario para guardar resultados de todos os teachers
resultados_fase1 = {}

# ── loop pelos tres teachers ─────────────────────────────────────────────────
for nome_teacher in ['resnet50', 'vgg16', 'convnext_small']:
    print(f"\n{'='*60}")
    print(f" teacher: {nome_teacher.upper()} | dataset: {DATASET_ESCOLHIDO}")
    print(f"{'='*60}")

    # cria o teacher e congela o encoder (passo obrigatorio)
    teacher = TeacherWrapper(nome_teacher, n_classes=N_CLASSES).to(device)
    teacher.freeze_encoder()

    # apenas os parametros do classificador sao atualizados
    params_treinaveis = [p for p in teacher.parameters() if p.requires_grad]
    print(f"  parametros treinaveis: {sum(p.numel() for p in params_treinaveis):,}")

    optimizer = optim.Adam(params_treinaveis, lr=LR_FASE1, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FASE1)

    historico = loop_treino(
        model        = teacher,
        loader_train = DATASETS[DATASET_ESCOLHIDO]['train'],
        loader_val   = DATASETS[DATASET_ESCOLHIDO]['val'],
        optimizer    = optimizer,
        scheduler    = scheduler,
        n_epochs     = EPOCHS_FASE1,
        device       = device,
        descricao    = nome_teacher,
    )

    # avalia no conjunto de teste
    accs_te = []
    for dados in DATASETS[DATASET_ESCOLHIDO]['test']:
        _, a = validar_batch(teacher, dados, device)
        accs_te.append(a)
    acc_teste = np.mean(accs_te)
    print(f"  acuracia TESTE: {acc_teste:.4f}")

    resultados_fase1[nome_teacher] = {
        'teacher_obj': teacher,
        'historico':   historico,
        'acc_teste':   acc_teste,
    }

    # salva o classificador (sera reutilizado exatamente na fase 3)
    os.makedirs("checkpoints", exist_ok=True)
    torch.save(teacher.classifier.state_dict(),
               f"checkpoints/classifier_{nome_teacher}_{DATASET_ESCOLHIDO}.pth")
    print(f"  classificador salvo em checkpoints/")
    plotar_curvas(historico, titulo=f"{nome_teacher} - fase 1")


In [ ]:
# tabela de comparacao: Q1 - qual teacher transfere melhor?

print("\nQ1 - Comparacao de Teachers (Fase 1)\n")
linhas = []
for nome, dados in resultados_fase1.items():
    linhas.append({'teacher': nome, 'acc_teste': dados['acc_teste']})

df_q1 = pd.DataFrame(linhas).sort_values('acc_teste', ascending=False)
df_q1['params_M'] = df_q1['teacher'].map({'vgg16': 138, 'resnet50': 25, 'convnext_small': 50})
df_q1['gflops']   = df_q1['teacher'].map({'vgg16': 15.5, 'resnet50': 4.1, 'convnext_small': 8.7})
print(df_q1.to_string(index=False))
print()
print("-> observacao esperada: accuracy nao e proporcional ao tamanho do teacher")
print("-> capacidade de transferencia depende da qualidade das representacoes")


---
## Secao 4 - Arquitetura do Student

### Objetivo: muito menor, quase igual em precisao

O student deve ser consideravelmente mais leve que o teacher:
- Menos de 10% dos GFLOPs do teacher
- Menos de 20% dos parametros do teacher

### Duas partes do student

```
Imagem
  |
[ENCODER DO STUDENT]
(CNN leve que aprende features proprias)
  |
Features: [B, C_s, H_s, W_s]
  |
[PREDITOR DO STUDENT]
(mapeia features student -> espaco do teacher)
  |
Predicao: [B, C_teacher]
```

### Q2 - Tipos de preditor

| Tipo | Alvo | Preditor | Vantagem |
|------|------|----------|----------|
| **post-GAP** | vetor [B, C_t] | MLP (denso) | simples, dimensao baixa |
| **pre-GAP** | mapa [B, C_t, 7, 7] | Conv + pool | preserva localizacao espacial |

### Q3 - Tres variantes de encoder

| Variante | Descricao | Custo |
|----------|-----------|-------|
| PlainCNN | Conv-BN-ReLU empilhados | baseline |
| DepthwiseCNN | Convolucao depthwise-separavel (MobileNet) | ~8x menos params |
| MiniResNet | Conv + skip connections | melhor fluxo de gradiente |


In [ ]:
# bloco de convolucao padrao (mesmo padrao do curso mo434)

def conv_block(c_in, c_out, stride=1, dw=False):
    # bloco de convolucao com batchnorm e relu
    # se dw=True: usa convolucao depthwise-separavel (muito mais eficiente)
    if dw:
        # depthwise separavel: 1 filtro por canal + mistura de canais separada
        return nn.Sequential(
            nn.Conv2d(c_in, c_in, 3, stride=stride, padding=1, groups=c_in, bias=False),
            nn.BatchNorm2d(c_in),
            nn.ReLU(),
            nn.Conv2d(c_in, c_out, 1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(),
        )
    else:
        return nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(),
        )


# ── encoder a: plain cnn ─────────────────────────────────────────────────────

class PlainCNNEncoder(nn.Module):
    # encoder leve: 4 conv blocks empilhados com stride 2 em cada
    def __init__(self, channels=(32, 64, 128, 256)):
        super().__init__()
        layers = [conv_block(3, channels[0], stride=2)]
        for i in range(1, len(channels)):
            layers.append(conv_block(channels[i-1], channels[i], stride=2))
        self.features = nn.Sequential(*layers)
        self.out_dim  = channels[-1]

    def forward(self, x):
        return self.features(x)   # saida: [B, C, H/16, W/16]


# ── encoder b: depthwise cnn (mobilenet-style) ───────────────────────────────

class DepthwiseCNNEncoder(nn.Module):
    # encoder com convolucos depthwise-separaveis: muito mais leve
    def __init__(self, channels=(32, 64, 128, 256)):
        super().__init__()
        # primeira camada: conv padrao (input rgb, depthwise nao faz sentido aqui)
        layers = [conv_block(3, channels[0], stride=2, dw=False)]
        for i in range(1, len(channels)):
            layers.append(conv_block(channels[i-1], channels[i], stride=2, dw=True))
        self.features = nn.Sequential(*layers)
        self.out_dim  = channels[-1]

    def forward(self, x):
        return self.features(x)


# ── encoder c: mini-resnet ────────────────────────────────────────────────────

class ResBlock(nn.Module):
    # bloco residual: y = F(x) + x (skip connection)
    # melhora o fluxo de gradiente em redes profundas
    def __init__(self, c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c), nn.ReLU(),
            nn.Conv2d(c, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.conv(x) + x)  # adiciona a entrada original (residual)


class MiniResNetEncoder(nn.Module):
    # encoder leve com skip connections
    def __init__(self, channels=(32, 64, 128, 256)):
        super().__init__()
        layers = [conv_block(3, channels[0], stride=2)]
        for i in range(1, len(channels)):
            layers.append(conv_block(channels[i-1], channels[i], stride=2))
            layers.append(ResBlock(channels[i]))  # bloco residual apos cada downsampling
        self.features = nn.Sequential(*layers)
        self.out_dim  = channels[-1]

    def forward(self, x):
        return self.features(x)


# verificacao de shapes e parametros
print("shapes e parametros dos encoders do student:\n")
dummy = torch.randn(2, 3, 224, 224)
STUDENT_CHANNELS = (32, 64, 128, 256)

for nome, Encoder in [('PlainCNN', PlainCNNEncoder),
                       ('Depthwise', DepthwiseCNNEncoder),
                       ('MiniResNet', MiniResNetEncoder)]:
    enc = Encoder(STUDENT_CHANNELS)
    with torch.no_grad():
        out = enc(dummy)
    params = sum(p.numel() for p in enc.parameters()) / 1e6
    print(f"  {nome:12s} | saida: {str(out.shape):24s} | params: {params:.2f}M")


In [ ]:
# preditores para post-gap e pre-gap (Q2)

class PreditorPostGAP(nn.Module):
    # mapeia features do student (apos gap) para vetor C_teacher-dimensional
    # arquitetura: mlp com camada oculta
    def __init__(self, dim_student: int, dim_teacher: int, dim_hidden: int = 512):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(dim_student, dim_hidden),
            nn.ReLU(),
            nn.Linear(dim_hidden, dim_teacher),  # projeta para espaco do teacher
        )

    def forward(self, features_student):
        # features_student: [B, C_s, H, W] -> predicao: [B, C_teacher]
        x = self.gap(features_student).flatten(1)
        return self.mlp(x)


class PreditorPreGAP(nn.Module):
    # mapeia features do student para mapa espacial do teacher [B, C_teacher, 7, 7]
    # arquitetura: conv 1x1 para expandir canais + adaptive pool para alinhar dims
    def __init__(self, dim_student: int, dim_teacher: int):
        super().__init__()
        self.expand = nn.Sequential(
            nn.Conv2d(dim_student, dim_teacher, kernel_size=1, bias=False),
            nn.BatchNorm2d(dim_teacher),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool2d(7)  # forca saida para 7x7 (igual ao teacher)

    def forward(self, features_student):
        # features_student: [B, C_s, H, W] -> predicao: [B, C_teacher, 7, 7]
        x = self.expand(features_student)
        return self.pool(x)


class StudentModel(nn.Module):
    # modelo student completo: encoder + preditor
    def __init__(self, encoder: nn.Module, preditor: nn.Module):
        super().__init__()
        self.encoder  = encoder
        self.preditor = preditor

    def forward(self, x):
        features = self.encoder(x)
        return self.preditor(features)


# verificacao
print("shapes dos preditores:\n")
dummy_feat = torch.randn(2, 256, 14, 14)
for nome_t, dim_t in [('resnet50', 2048), ('vgg16', 512), ('convnext_small', 768)]:
    pred_post = PreditorPostGAP(256, dim_t)
    pred_pre  = PreditorPreGAP(256, dim_t)
    with torch.no_grad():
        out_post = pred_post(dummy_feat)
        out_pre  = pred_pre(dummy_feat)
    print(f"  teacher={nome_t:15s} | post-gap: {str(out_post.shape):16s} | pre-gap: {str(out_pre.shape)}")


---
## Secao 5 - Funcoes de Perda (Q4 e Q5)

### Q4 - Combinacao MSE + Cross-Entropy

A perda total combina dois objetivos:

$$L_{total} = \alpha \cdot L_{MSE} + (1 - \alpha) \cdot L_{CE}$$

- **$L_{MSE}$**: alinha features do student com features do teacher
- **$L_{CE}$**: mantém previsoes alinhadas com rotulos verdadeiros
- **$\alpha$**: hiperparametro para balancear os dois objetivos

| alpha | Significado |
|-------|-------------|
| 1.0 | so MSE (destilacao pura, sem rotulos) |
| 0.0 | so CE (ignora teacher) |
| 0.7 | ponto de partida recomendado |

### Q5 - Relational Knowledge Distillation (RKD)

Ao inves de alinhar **valores absolutos** de features, o RKD alinha as
**relacoes entre amostras** (distancias par-a-par):

> "Amostras similares no espaco do teacher devem ser similares no espaco do student"
> (Park et al., CVPR 2019)

Isso e **invariante a escala** das features e pode ser mais robusto quando
o student tem capacidade menor que o teacher (capacity mismatch).


In [ ]:
# funcao de perda combinada mse + cross-entropy (Q4)

class PerdaKD(nn.Module):
    # perda de destilacao de conhecimento:
    # L = alpha * MSE(pred_student, feat_teacher) + (1-alpha) * CE(logits, labels)
    def __init__(self, alpha: float = 0.7):
        super().__init__()
        self.alpha = alpha

    def forward(self, pred_student, feat_teacher, logits_via_classifier, labels):
        # pred_student: saida do preditor do student
        # feat_teacher: features do teacher (alvo)
        # logits_via_classifier: predicao passada pelo classificador do teacher
        # labels: rotulos verdadeiros
        l_mse = F.mse_loss(pred_student, feat_teacher)
        l_ce  = F.cross_entropy(logits_via_classifier, labels)
        perda = self.alpha * l_mse + (1 - self.alpha) * l_ce
        return perda, l_mse.item(), l_ce.item()


# relational knowledge distillation - rkd (Park et al., CVPR 2019)
class PerdaRKD(nn.Module):
    # ao inves de alinhar features absolutas, alinha distancias par-a-par
    # isso e invariante a escala e captura a geometria do espaco de features
    # referencia: https://arxiv.org/abs/1904.05068
    def __init__(self, weight_dist=1.0, weight_angle=2.0):
        super().__init__()
        self.w_dist  = weight_dist
        self.w_angle = weight_angle

    def forward(self, feat_student, feat_teacher):
        # feat_student e feat_teacher: [B, D] (features pos-gap)
        l_dist  = self._distancia(feat_student, feat_teacher)
        l_angle = self._angulo(feat_student, feat_teacher)
        return self.w_dist * l_dist + self.w_angle * l_angle

    def _distancia(self, fs, ft):
        # normaliza distancias par-a-par e compara (invariante a escala)
        d_s = torch.cdist(fs, fs)
        d_t = torch.cdist(ft, ft)
        d_s = d_s / (d_s.mean() + 1e-8)  # normaliza pela media
        d_t = d_t / (d_t.mean() + 1e-8)
        return F.huber_loss(d_s, d_t)    # huber e robusto a outliers

    def _angulo(self, fs, ft):
        # compara similaridade coseno par-a-par (geometria angular)
        fs_norm = F.normalize(fs, dim=1)  # projeta na esfera unitaria
        ft_norm = F.normalize(ft, dim=1)
        sim_s = fs_norm @ fs_norm.T       # similaridade coseno: [B, B]
        sim_t = ft_norm @ ft_norm.T
        return F.mse_loss(sim_s, sim_t)


# teste rapido das funcoes de perda
print("testando funcoes de perda:")
pred_s = torch.randn(8, 2048)
feat_t = torch.randn(8, 2048)
logits = torch.randn(8, 100)
labels = torch.randint(0, 100, (8,))

perda_kd  = PerdaKD(alpha=0.7)
perda_rkd = PerdaRKD()

l_total, l_mse, l_ce = perda_kd(pred_s, feat_t, logits, labels)
l_rkd = perda_rkd(pred_s, feat_t)

print(f"  perda kd  (alpha=0.7): total={l_total:.4f} | mse={l_mse:.4f} | ce={l_ce:.4f}")
print(f"  perda rkd:             {l_rkd:.4f}")


---
## Secao 6 - Fase 2: Treinar o Student por Destilacao

### Como funciona a Fase 2?

O teacher esta completamente congelado. O student aprende a imitar
as features do teacher atraves do gradiente da perda de destilacao.

```
Imagem -> [TEACHER ENCODER, congelado] -> features_teacher (alvo, sem gradiente)
Imagem -> [STUDENT ENCODER, treina]    -> features_student_raw
       -> [PREDITOR, treina]           -> pred_features
                                             |
                                  MSE(pred_features, features_teacher)
                                  + CE(classifier(pred_features), labels)
```

### Ponto critico: `torch.no_grad()` no teacher

Ao extrair features do teacher, NUNCA deixe gradientes fluirem pelo encoder.
Use sempre `with torch.no_grad():` para economizar memoria e tempo de computo.


In [ ]:
# loop de treino da fase 2 (destilacao)

def treinar_batch_kd(student, teacher, dados, optimizer, perda_fn, device, modo_target='post_gap'):
    # um passo de treino de destilacao:
    #   1. extrai features do teacher (sem gradiente)
    #   2. student prediz essas features
    #   3. calcula perda combinada mse + ce
    #   4. backpropaga e atualiza apenas pesos do student
    student.train()
    teacher.eval()  # teacher sempre em eval (batchnorm usa estatisticas fixas)

    imgs, labels = dados
    imgs, labels = imgs.to(device), labels.to(device)

    # extrai targets do teacher sem gradiente (economiza memoria)
    with torch.no_grad():
        if modo_target == 'post_gap':
            feat_teacher = teacher.get_post_gap(imgs)   # [B, C_t]
        else:
            feat_teacher = teacher.get_pre_gap(imgs)    # [B, C_t, 7, 7]

    optimizer.zero_grad()
    pred_student = student(imgs)   # [B, C_t] ou [B, C_t, 7, 7]

    # passa predicao pelo classificador do teacher para obter logits
    with torch.no_grad():
        if modo_target == 'post_gap':
            logits = teacher.classifier(pred_student)
        else:
            pooled = teacher.gap(pred_student).flatten(1)
            logits = teacher.classifier(pooled)

    perda, l_mse, l_ce = perda_fn(pred_student, feat_teacher, logits, labels)
    perda.backward()
    # clipa gradientes para evitar explosao (importante para redes profundas)
    torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=1.0)
    optimizer.step()

    acc = (logits.argmax(1) == labels).float().mean().item()
    return perda.item(), l_mse, l_ce, acc


@torch.no_grad()
def validar_student(student, teacher, loader_val, device, modo_target='post_gap'):
    # avalia o student usando o classificador do teacher (preparacao para fase 3)
    student.eval(); teacher.eval()
    accs = []
    for imgs, labels in loader_val:
        imgs, labels = imgs.to(device), labels.to(device)
        pred = student(imgs)
        if modo_target == 'post_gap':
            logits = teacher.classifier(pred)
        else:
            pooled = teacher.gap(pred).flatten(1)
            logits = teacher.classifier(pooled)
        accs.append((logits.argmax(1) == labels).float().mean().item())
    return np.mean(accs)


def loop_destilacao(student, teacher, loader_train, loader_val, perda_fn,
                    n_epochs, lr, device, modo_target='post_gap', descricao=''):
    # loop completo de destilacao com registro de metricas
    optimizer = optim.AdamW(student.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    historico  = defaultdict(list)
    melhor_acc = 0.0
    melhor_pesos = None

    for epoch in range(1, n_epochs + 1):
        perdas, mselist, celist, accs_tr = [], [], [], []
        for dados in tqdm(loader_train, desc=f"[{descricao}] ep.{epoch}", leave=False):
            p, m, c, a = treinar_batch_kd(student, teacher, dados, optimizer,
                                           perda_fn, device, modo_target)
            perdas.append(p); mselist.append(m); celist.append(c); accs_tr.append(a)

        acc_vl = validar_student(student, teacher, loader_val, device, modo_target)
        scheduler.step()

        historico['perda'].append(np.mean(perdas))
        historico['mse'].append(np.mean(mselist))
        historico['acc_tr'].append(np.mean(accs_tr))
        historico['acc_vl'].append(acc_vl)

        if acc_vl > melhor_acc:
            melhor_acc   = acc_vl
            melhor_pesos = deepcopy(student.state_dict())

        if epoch % 5 == 0 or epoch == 1:
            print(f"  ep.{epoch:3d} | mse={np.mean(mselist):.4f} | "
                  f"acc_tr={np.mean(accs_tr):.3f} | acc_vl={acc_vl:.3f}")

    if melhor_pesos:
        student.load_state_dict(melhor_pesos)
    print(f"  melhor acc validacao: {melhor_acc:.4f}")
    return historico, melhor_acc


In [ ]:
# experimento principal fase 2: responde Q1, Q2, Q3 e Q4

EPOCHS_FASE2  = 30      # aumente para 50 no experimento final
LR_FASE2      = 1e-3
DATASET_FASE2 = 'cifar100'

# configuracoes do experimento (teacher, encoder, modo_target, alpha)
CONFIGS_EXPERIMENTO = [
    # Q3: comparar tres encoders com mesmo teacher e alpha
    ('resnet50', PlainCNNEncoder,    'post_gap', 0.7),
    ('resnet50', DepthwiseCNNEncoder,'post_gap', 0.7),
    ('resnet50', MiniResNetEncoder,  'post_gap', 0.7),
    # Q2: comparar pre-gap vs post-gap com mesmo encoder
    ('resnet50', PlainCNNEncoder,    'pre_gap',  0.7),
    # Q1: comparar outros teachers
    ('vgg16',          PlainCNNEncoder, 'post_gap', 0.7),
    ('convnext_small', PlainCNNEncoder, 'post_gap', 0.7),
    # Q4: ablacao de alpha (so MSE, so CE, combinacoes)
    ('resnet50', PlainCNNEncoder, 'post_gap', 1.0),
    ('resnet50', PlainCNNEncoder, 'post_gap', 0.5),
    ('resnet50', PlainCNNEncoder, 'post_gap', 0.0),
]

resultados_fase2 = []

for nome_teacher, EncoderCls, modo_target, alpha in CONFIGS_EXPERIMENTO:
    descricao = f"{nome_teacher[:3]}_{EncoderCls.__name__[:5]}_{modo_target[:3]}_a{alpha}"
    print(f"\n{'─'*62}")
    print(f" teacher={nome_teacher} | encoder={EncoderCls.__name__}")
    print(f" target={modo_target} | alpha={alpha}")
    print(f"{'─'*62}")

    # recupera teacher ja treinado na fase 1
    teacher = resultados_fase1[nome_teacher]['teacher_obj'].to(device)

    # constroi o student
    encoder  = EncoderCls(STUDENT_CHANNELS)
    dim_t    = teacher.feat_dim
    preditor = (PreditorPostGAP(encoder.out_dim, dim_t)
                if modo_target == 'post_gap'
                else PreditorPreGAP(encoder.out_dim, dim_t))
    student = StudentModel(encoder, preditor).to(device)

    # relatorio de eficiencia (Q3)
    params_s = sum(p.numel() for p in student.parameters()) / 1e6
    params_t = sum(p.numel() for p in teacher.parameters()) / 1e6
    print(f"  params student: {params_s:.2f}M | teacher: {params_t:.2f}M ({params_s/params_t*100:.1f}%)")

    # treina por destilacao
    perda_fn = PerdaKD(alpha=alpha)
    historico, melhor_acc = loop_destilacao(
        student      = student,
        teacher      = teacher,
        loader_train = DATASETS[DATASET_FASE2]['train'],
        loader_val   = DATASETS[DATASET_FASE2]['val'],
        perda_fn     = perda_fn,
        n_epochs     = EPOCHS_FASE2,
        lr           = LR_FASE2,
        device       = device,
        modo_target  = modo_target,
        descricao    = descricao,
    )

    resultados_fase2.append({
        'descricao':   descricao,
        'teacher':     nome_teacher,
        'encoder':     EncoderCls.__name__,
        'modo_target': modo_target,
        'alpha':       alpha,
        'melhor_acc':  melhor_acc,
        'params_M':    params_s,
        'historico':   historico,
        'student_obj': student,
    })

    os.makedirs("checkpoints", exist_ok=True)
    torch.save(student.state_dict(), f"checkpoints/student_{descricao}.pth")


---
## Secao 7 - Q5: Comparacao com a Literatura (RKD)

### Por que RKD pode superar o MSE baseline?

O MSE forca o student a copiar os **valores absolutos** das features do teacher.
Isso pode ser dificil quando o student tem capacidade menor (capacity mismatch).

O RKD alinha as **relacoes entre amostras** ao inves de valores absolutos:
> "Amostras similares no espaco do teacher devem ser similares no espaco do student"

Essa abordagem e invariante a escala das features e captura a geometria
do espaco de representacao sem exigir alinhamento perfeito ponto-a-ponto.


In [ ]:
# comparacao mse-baseline vs rkd (Q5)

def treinar_batch_rkd(student, teacher, dados, optimizer, perda_rkd_fn, alpha_ce, device):
    # passo de treino com rkd:
    # l_total = rkd(feat_student, feat_teacher) + alpha_ce * ce(logits, labels)
    student.train(); teacher.eval()
    imgs, labels = dados
    imgs, labels = imgs.to(device), labels.to(device)

    with torch.no_grad():
        feat_teacher = teacher.get_post_gap(imgs)

    optimizer.zero_grad()
    pred_student = student(imgs)

    # rkd compara relacoes par-a-par no espaco de features
    l_rkd = perda_rkd_fn(pred_student, feat_teacher)

    # ce supervisionado para nao perder alinhamento com rotulos
    with torch.no_grad():
        logits = teacher.classifier(pred_student)
    l_ce = F.cross_entropy(logits, labels)

    perda = l_rkd + alpha_ce * l_ce
    perda.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    optimizer.step()
    acc = (logits.argmax(1) == labels).float().mean().item()
    return perda.item(), l_rkd.item(), l_ce.item(), acc


def comparar_mse_vs_rkd(teacher_nome, dataset_nome, n_epochs=20, device=device):
    # compara mse baseline vs rkd com mesmo encoder e teacher
    teacher = resultados_fase1[teacher_nome]['teacher_obj'].to(device)
    resultados_comp = {}

    for nome_perda, modo in [('MSE_baseline', 'mse'), ('RKD', 'rkd')]:
        print(f"  treinando: {nome_perda}")

        enc    = PlainCNNEncoder(STUDENT_CHANNELS)
        pred   = PreditorPostGAP(enc.out_dim, teacher.feat_dim)
        stu    = StudentModel(enc, pred).to(device)
        optim_ = optim.AdamW(stu.parameters(), lr=1e-3, weight_decay=1e-4)
        sched_ = optim.lr_scheduler.CosineAnnealingLR(optim_, T_max=n_epochs)

        perda_fn_mse = PerdaKD(alpha=0.7)
        perda_fn_rkd = PerdaRKD()
        accs_vl = []

        for epoch in range(1, n_epochs + 1):
            for dados in DATASETS[dataset_nome]['train']:
                if modo == 'mse':
                    treinar_batch_kd(stu, teacher, dados, optim_, perda_fn_mse, device)
                else:
                    treinar_batch_rkd(stu, teacher, dados, optim_, perda_fn_rkd, 0.3, device)

            acc = validar_student(stu, teacher, DATASETS[dataset_nome]['val'], device)
            accs_vl.append(acc)
            sched_.step()

        resultados_comp[nome_perda] = {'accs_vl': accs_vl, 'melhor_acc': max(accs_vl)}
        print(f"    melhor acc: {max(accs_vl):.4f}")

    # plota comparacao
    plt.figure(figsize=(8, 4))
    for nome, res in resultados_comp.items():
        plt.plot(res['accs_vl'], label=f"{nome} (melhor={res['melhor_acc']:.3f})")
    plt.xlabel('epoca'); plt.ylabel('acuracia validacao')
    plt.title(f'MSE vs RKD | teacher={teacher_nome}, dataset={dataset_nome}')
    plt.legend(); plt.grid(True, alpha=0.3); plt.show()
    return resultados_comp


# descomente para executar:
# resultados_q5 = comparar_mse_vs_rkd('resnet50', 'cifar100', n_epochs=20)


---
## Secao 8 - Fase 3: Avaliacao Final

### O teste definitivo

Na Fase 3, o **classificador da Fase 1** (congelado) e usado para avaliar
o student no conjunto de **teste**. Isso mede diretamente se o student
aprendeu a produzir features no mesmo espaco do teacher.

Se o student aprendeu bem as representacoes, o classificador do teacher
conseguira classificar as predicoes do student com alta acuracia, mesmo
nunca tendo visto as features do student durante o treinamento.


In [ ]:
# avaliacao final na fase 3

@torch.no_grad()
def avaliar_fase3(student, teacher, loader_test, device, modo_target='post_gap'):
    # avaliacao definitiva: usa classificador do teacher para avaliar student
    # nunca viu as features do student antes (transferencia real de conhecimento)
    student.eval(); teacher.eval()
    accs, top5_accs = [], []

    for imgs, labels in loader_test:
        imgs, labels = imgs.to(device), labels.to(device)
        pred = student(imgs)

        if modo_target == 'post_gap':
            logits = teacher.classifier(pred)
        else:
            pooled = teacher.gap(pred).flatten(1)
            logits = teacher.classifier(pooled)

        # top-1
        accs.append((logits.argmax(1) == labels).float().mean().item())

        # top-5 (mais informativo para cifar-100 com 100 classes)
        if logits.shape[1] >= 5:
            top5 = logits.topk(5, dim=1).indices
            top5_accs.append((top5 == labels.unsqueeze(1)).any(1).float().mean().item())

    return np.mean(accs), (np.mean(top5_accs) if top5_accs else None)


# avalia todos os experimentos no conjunto de teste
print("Avaliacao Fase 3 - todos os experimentos\n")
print(f"{'descricao':45s} {'acc_val':>8s} {'acc_test':>9s} {'params_M':>9s}")
print("─" * 76)

for exp in resultados_fase2:
    teacher = resultados_fase1[exp['teacher']]['teacher_obj'].to(device)
    acc_test, acc_top5 = avaliar_fase3(
        exp['student_obj'], teacher,
        DATASETS[DATASET_FASE2]['test'],
        device, exp['modo_target']
    )
    exp['acc_test'] = acc_test
    top5_str = f" (top5={acc_top5:.3f})" if acc_top5 else ""
    print(f"  {exp['descricao']:43s} {exp['melhor_acc']:>8.4f} {acc_test:>9.4f} {exp['params_M']:>9.2f}M{top5_str}")


In [ ]:
# tabela final com respostas para cada questao de pesquisa

df_resultados = pd.DataFrame([
    {
        'teacher':     r['teacher'],
        'encoder':     r['encoder'],
        'target':      r['modo_target'],
        'alpha':       r['alpha'],
        'acc_val':     r['melhor_acc'],
        'acc_test':    r.get('acc_test', 0),
        'params_M':    r['params_M'],
    }
    for r in resultados_fase2
])

# Q1
print("=" * 55)
print(" Q1 - QUAL TEACHER TRANSFERE MELHOR?")
print("=" * 55)
q1 = df_resultados[
    (df_resultados['encoder'] == 'PlainCNNEncoder') &
    (df_resultados['target']  == 'post_gap') &
    (df_resultados['alpha']   == 0.7)
][['teacher', 'acc_test']].sort_values('acc_test', ascending=False)
print(q1.to_string(index=False))

# Q2
print("\n" + "=" * 55)
print(" Q2 - PRE-GAP VS POST-GAP?")
print("=" * 55)
q2 = df_resultados[
    (df_resultados['teacher'] == 'resnet50') &
    (df_resultados['encoder'] == 'PlainCNNEncoder') &
    (df_resultados['alpha']   == 0.7)
][['target', 'acc_test']].sort_values('acc_test', ascending=False)
print(q2.to_string(index=False))

# Q3
print("\n" + "=" * 55)
print(" Q3 - MELHOR ARQUITETURA DO STUDENT?")
print("=" * 55)
q3 = df_resultados[
    (df_resultados['teacher'] == 'resnet50') &
    (df_resultados['target']  == 'post_gap') &
    (df_resultados['alpha']   == 0.7)
][['encoder', 'acc_test', 'params_M']].sort_values('acc_test', ascending=False)
print(q3.to_string(index=False))

# Q4
print("\n" + "=" * 55)
print(" Q4 - MELHOR FUNCAO DE PERDA?")
print("=" * 55)
q4 = df_resultados[
    (df_resultados['teacher'] == 'resnet50') &
    (df_resultados['encoder'] == 'PlainCNNEncoder') &
    (df_resultados['target']  == 'post_gap')
][['alpha', 'acc_test']].sort_values('alpha')
print(q4.to_string(index=False))


In [ ]:
# graficos de analise final

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Q1: comparacao de teachers
ax = axes[0]
dados_q1 = df_resultados[
    (df_resultados['encoder'] == 'PlainCNNEncoder') &
    (df_resultados['target']  == 'post_gap') &
    (df_resultados['alpha']   == 0.7)
].sort_values('acc_test', ascending=True)
ax.barh(dados_q1['teacher'], dados_q1['acc_test'], color='steelblue', alpha=0.8)
ax.set_title('Q1 - Qual teacher transfere melhor?', fontsize=11)
ax.set_xlabel('acuracia no teste'); ax.grid(True, alpha=0.3, axis='x')

# Q4: ablacao de alpha
ax = axes[1]
dados_q4 = df_resultados[
    (df_resultados['teacher'] == 'resnet50') &
    (df_resultados['encoder'] == 'PlainCNNEncoder') &
    (df_resultados['target']  == 'post_gap')
].sort_values('alpha')
ax.plot(dados_q4['alpha'], dados_q4['acc_test'], 'o-', color='darkorange', linewidth=2)
ax.set_title('Q4 - Ablacao do peso alpha', fontsize=11)
ax.set_xlabel('alpha (peso do MSE)'); ax.set_ylabel('acuracia no teste')
ax.grid(True, alpha=0.3)

# Q3: acuracia vs parametros
ax = axes[2]
dados_q3 = df_resultados[
    (df_resultados['teacher'] == 'resnet50') &
    (df_resultados['target']  == 'post_gap') &
    (df_resultados['alpha']   == 0.7)
]
for _, row in dados_q3.iterrows():
    ax.scatter(row['params_M'], row['acc_test'], s=120, zorder=5)
    label = row['encoder'].replace('Encoder', '').replace('CNN', '')
    ax.annotate(label, (row['params_M'], row['acc_test']),
                textcoords='offset points', xytext=(5, 3), fontsize=9)
ax.set_title('Q3 - Trade-off: acuracia vs parametros', fontsize=11)
ax.set_xlabel('parametros do student (M)'); ax.set_ylabel('acuracia no teste')
ax.grid(True, alpha=0.3)

plt.suptitle('Analise dos Experimentos - Knowledge Distillation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


---
## Secao 9 - Metricas de Eficiencia

Uma das metricas mais importantes e mostrar que o student e **muito mais eficiente**
que o teacher. Use `fvcore` para calcular GFLOPs automaticamente.


In [ ]:
# calculo de gflops e parametros

try:
    from fvcore.nn import FlopCountAnalysis

    def contar_eficiencia(model, input_size=(1, 3, 224, 224)):
        # retorna (gflops, n_parametros_M) de um modelo
        model.eval()
        dummy = torch.randn(*input_size)
        try:
            flops  = FlopCountAnalysis(model, dummy)
            gflops = flops.total() / 1e9
        except Exception:
            gflops = None
        params = sum(p.numel() for p in model.parameters()) / 1e6
        return gflops, params

    print("Eficiencia Computacional\n")
    print(f"{'modelo':30s} {'gflops':>8s} {'params_M':>10s} {'reducao':>12s}")
    print("─" * 65)

    # teacher de referencia
    t_ref    = resultados_fase1['resnet50']['teacher_obj'].cpu()
    gf_t, pr_t = contar_eficiencia(t_ref)
    print(f"  {'resnet50 (teacher)':28s} {gf_t or 'n/d':>8} {pr_t:>10.1f}M {'(ref)':>12s}")

    # students
    for exp in resultados_fase2:
        if exp['teacher'] == 'resnet50' and exp['alpha'] == 0.7 and exp['modo_target'] == 'post_gap':
            stu_cpu = exp['student_obj'].cpu()
            gf_s, pr_s = contar_eficiencia(stu_cpu)
            reducao    = f"{pr_s/pr_t*100:.1f}%"
            gf_str     = f"{gf_s:.3f}" if gf_s else "n/d"
            print(f"  {exp['encoder']:28s} {gf_str:>8s} {pr_s:>10.2f}M {reducao:>12s}")

except ImportError:
    print("fvcore nao instalado. execute: pip install fvcore")
    print("\nParametros (calculo manual):")
    for exp in resultados_fase2:
        if exp['alpha'] == 0.7 and exp['modo_target'] == 'post_gap':
            print(f"  {exp['encoder']:30s} {exp['params_M']:.2f}M")


---
## Secao 10 - Checklist Final e Proximos Passos

### O que voce ja implementou

- [x] Dois datasets (CIFAR-100 + STL-10)
- [x] Tres teachers (VGG-16, ResNet-50, ConvNeXt-Small)
- [x] Fase 1: treino do classificador com encoder congelado
- [x] Tres variantes de encoder do student (Plain, Depthwise, MiniResNet)
- [x] Dois preditores (post-GAP e pre-GAP)
- [x] Fase 2: loop de destilacao com MSE + CE
- [x] Ablacao de alpha (Q4)
- [x] Perda RKD para Q5
- [x] Fase 3: avaliacao com classificador fixo
- [x] Tabelas e graficos de analise

### Para completar o relatorio (responda cada questao)

| Questao | O que reportar |
|---------|----------------|
| Q1 | Tabela: teacher x acc_test nos 2 datasets |
| Q2 | Tabela: pre-GAP vs post-GAP; discuta trade-off espacial vs simplicidade |
| Q3 | Tabela: encoder x acc_test x GFLOPs x params |
| Q4 | Grafico: alpha x acc_test; identifique o alpha otimo |
| Q5 | Tabela: MSE vs RKD x acc_test; discuta quando RKD supera MSE |

### Melhorias para o experimento final

```python
# aumentar epocas (recomendado para resultados finais)
EPOCHS_FASE1 = 20
EPOCHS_FASE2 = 50

# repetir para o segundo dataset
for dataset_nome in ['cifar100', 'stl10']:
    for nome_teacher in ['resnet50', 'vgg16', 'convnext_small']:
        ...

# mixed precision para treinar mais rapido na GPU
from torch.cuda.amp import GradScaler, autocast
scaler = GradScaler()
with autocast():
    pred  = student(imgs)
    perda = perda_fn(...)
scaler.scale(perda).backward()
scaler.step(optimizer)
scaler.update()
```

### Referencias

1. Hinton et al. (2015) - Distilling the Knowledge in a Neural Network [arXiv:1503.02531]
2. Romero et al. (2015) - FitNets: Hints for Thin Deep Nets [ICLR 2015]
3. Zagoruyko & Komodakis (2017) - Paying More Attention to Attention [ICLR 2017]
4. Park et al. (2019) - Relational Knowledge Distillation [CVPR 2019]
5. Cooper et al. (2025) - Logit-Based Losses Limit Feature KD [OpenReview]
6. Goodfellow, Bengio & Courville (2016) - Deep Learning [MIT Press]


In [ ]:
### CRIAR CLASSES PARA ESSE NOTEBOOK 
### rodar o claude com 4.7 e advanced thinking para criar um .py com as classes e funcoes principais
### (modelos, perdas, loop de treino) para deixar o notebook mais limpo e focado na analise dos resultados

In [ ]:
### CRIAR NOTEBOOKS SEPARADOS PARA CADA FASE para ir testando passo a passo 

In [ ]:
### REVER O CODIGO PARA DEIXAR MAIS CLARO E ORGANIZADO (EX: SEPARAR FUNCOES DE TREINO, MODELOS, PERDAS, ETC)

In [ ]:
### rever o jeito que o codigo comenta as coisas e ajustar 

In [ ]:
### entender que passos precisa de explicacoes mais matematicas e quais precisam de mais explicacoes intuitivas (ex: o que e rkd, porque usar mse + ce, etc)